#### Objective: Đánh giá hiệu quả của chiến dịch voucher (treatment) lên nhóm khách hàng mục tiêu Cụm 2 
Hypothesis:
- H0: Voucher không làm thay đổi số chuyến đi trung bình của khách hàng thuộc Cluster 2.
- H1: Voucher làm tăng số chuyến đi trung bình của khách hàng thuộc Cluster 2.

Primary Metric: Incremental Trips per Rider

In [1]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

B    = r"c:/Users/Linh/Desktop/Growth & Experimentation Project for Ride-Hailing Promotions"
SRC  = os.path.join(B, "02. Synthetic_data", "outputs", "experiment_ab_final.csv")
SEG  = os.path.join(B, "03. Segmentation", "outputs", "rider_cluster.csv")
OUT  = os.path.join(B, "04. AB Testing", "outputs"); os.makedirs(OUT, exist_ok=True)

SURFACE, INK, SECOND, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SERIES, ALT     = "#e1e0d9", "#c3c2b7", "#2a78d6", "#eb6834"


df  = pd.read_csv(SRC)
seg = pd.read_csv(SEG)[["user_id", "cluster", "persona"]]

_before = len(df)
df = df.merge(seg, on="user_id", how="inner", validate="1:1")
assert len(df) == _before, f"merge mat dong: {_before:,} -> {len(df):,}"

In [2]:
TARGET_CLUSTER = 2
tg = df[df.cluster == TARGET_CLUSTER].copy()
TARGET = tg.persona.mode().iloc[0]          # ten mo ta, chi de hien thi
t_, c_ = tg[tg.T_rct == 1], tg[tg.T_rct == 0]

print(f"Cum {TARGET_CLUSTER} — {TARGET}")
print(f"   quy mo    : {len(tg):,} rider ({len(tg)/len(df):.1%} ")
print(f"   treatment : {len(t_):,}")
print(f"   control   : {len(c_):,}")

Cum 2 — Khach it gan ket, di theo lo trinh co dinh
   quy mo    : 4,513 rider (22.6% 
   treatment : 2,242
   control   : 2,271


In [3]:
COV = ["total_rides", "recency_days", "typical_distance", "route_entropy",
       "pct_airport", "weekend_ratio", "pct_flex_payment", "resid_tip_rate",
       "avg_fare", "age", "is_urban"]

def smd(a, b):
    """Standardized Mean Difference — chenh lech chia do lech chuan gop.
    Dung SMD thay vi p-value vi p-value phu thuoc co mau: n lon thi chenh
    lech vun vat cung 'co y nghia', n nho thi lech that cung khong bat duoc."""
    s = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return 0.0 if s == 0 else (a.mean() - b.mean()) / s

bal = pd.DataFrame({
    "treatment": [t_[c].mean() for c in COV],
    "control":   [c_[c].mean() for c in COV],
    "SMD":       [smd(t_[c], c_[c]) for c in COV],
}, index=COV)
bal["|SMD|"] = bal.SMD.abs()
display(bal.round(4).sort_values("|SMD|", ascending=False))

,treatment,control,SMD,|SMD|
weekend_ratio,0.2310,0.2563,-0.0771,0.0771
recency_days,56.3515,54.7587,0.0306,0.0306
route_entropy,0.6714,0.6582,0.0295,0.0295
pct_flex_payment,0.0053,0.0061,-0.0199,0.0199
pct_airport,0.0234,0.0248,-0.0157,0.0157
resid_tip_rate,-0.1230,-0.1219,-0.0154,0.0154
age,41.2217,41.4038,-0.0130,0.0130
is_urban,0.8715,0.8758,-0.0129,0.0129
avg_fare,18.7514,18.8661,-0.0114,0.0114
typical_distance,3.0939,3.1177,-0.0089,0.0089
